# Weather Intelligence: Sync, Embed, and Verify

This notebook runs the complete pre-deployment pipeline with plain Python and psycopg2:

1. Connect to Lakebase and confirm the pgvector schema.
2. Harvest National Weather Service alerts and forecasts.
3. Upsert normalized documents into `weather_documents`.
4. Chunk and embed new or changed documents.
5. Batch-write `VECTOR(384)` values into `weather_embeddings`.
6. Run a semantic-search smoke test before deploying the Flask app.

## 1. Install notebook dependencies
The embedding write path uses psycopg2 only; it does not use Spark JDBC.

In [ ]:
%pip install -q 'databricks-sdk>=0.57.0' 'psycopg2-binary>=2.9.10' 'requests>=2.32.3' 'sentence-transformers>=5.0.0,<6'

In [ ]:
dbutils.library.restartPython()

## 2. Configure the run
Change the widgets without editing pipeline code. Locations are separated by semicolons.

In [ ]:
dbutils.widgets.text("locations", "Chicago, IL;Austin, TX", "Locations (; separated)")
dbutils.widgets.text("documents_per_location", "50", "Documents per location")
dbutils.widgets.text("document_limit", "1000", "Maximum documents to embed")
dbutils.widgets.text("batch_size", "64", "Embedding write batch size")
dbutils.widgets.text("chunk_size", "800", "Chunk size (characters)")
dbutils.widgets.text("chunk_overlap", "100", "Chunk overlap (characters)")
dbutils.widgets.text("search_query", "flash flood risk this weekend", "Smoke-test query")

LOCATIONS = [value.strip() for value in dbutils.widgets.get("locations").split(";") if value.strip()]
DOCUMENTS_PER_LOCATION = max(1, min(int(dbutils.widgets.get("documents_per_location")), 200))
DOCUMENT_LIMIT = max(1, int(dbutils.widgets.get("document_limit")))
BATCH_SIZE = max(1, int(dbutils.widgets.get("batch_size")))
CHUNK_SIZE = max(1, int(dbutils.widgets.get("chunk_size")))
CHUNK_OVERLAP = int(dbutils.widgets.get("chunk_overlap"))
SEARCH_QUERY = dbutils.widgets.get("search_query").strip()

if not LOCATIONS:
    raise ValueError("Provide at least one location")
if not 0 <= CHUNK_OVERLAP < CHUNK_SIZE:
    raise ValueError("chunk_overlap must be smaller than chunk_size")

print(f"Locations: {LOCATIONS}")
print(f"Chunking: {CHUNK_SIZE} characters with {CHUNK_OVERLAP} overlap")

## 3. Import the shared project modules
The notebook and deployed app use the same connection, normalization, chunking, and write code.

In [ ]:
import sys
from pathlib import Path

candidate_roots = [Path.cwd(), Path.cwd().parent]
REPO_ROOT = next((path for path in candidate_roots if (path / "lakebase.py").exists()), None)
if REPO_ROOT is None:
    raise RuntimeError("Could not locate lakebase.py. Run this notebook from its Git repository.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import lakebase
from embedding_model import MODEL_NAME, embed_texts, vector_literal
from scripts.ingest_weather_embeddings import (
    build_rows,
    fetch_documents_needing_embeddings,
    replace_embeddings,
)
from weather_client import WeatherClient
from weather_store import upsert_weather_documents

print(f"Repository root: {REPO_ROOT}")
print(f"Embedding model: {MODEL_NAME}")

## 4. Connect to Lakebase and ensure the schema
You may create the tables first with `sql/01_setup_weather_vector_schema.sql`. This call is idempotent and confirms they exist.

In [ ]:
lakebase.ensure_weather_schema()
connection_info = lakebase.run_query(
    """
    SELECT current_database() AS database_name,
           current_user AS database_role,
           extversion AS pgvector_version
    FROM pg_extension
    WHERE extname = 'vector'
    """
)
display(connection_info)

## 5. Harvest and upsert weather narratives
Each location is resolved to an NWS grid point. Active alerts and detailed forecast periods are stored with their raw JSON payloads.

In [ ]:
client = WeatherClient()
sync_results = []
for location in LOCATIONS:
    documents = client.fetch_documents(location, limit=DOCUMENTS_PER_LOCATION)
    synced = upsert_weather_documents(documents)
    sync_results.append({"requested_location": location, "documents_synced": synced})

display(sync_results)
document_summary = lakebase.run_query(
    """
    SELECT location, source_type, COUNT(*) AS document_count, MAX(synced_at) AS last_synced_at
    FROM weather_documents
    GROUP BY location, source_type
    ORDER BY location, source_type
    """
)
display(document_summary)

## 6. Find new or changed documents

In [ ]:
documents = fetch_documents_needing_embeddings(DOCUMENT_LIMIT)
print(f"Documents awaiting embeddings: {len(documents)}")
display(documents[:10])

## 7. Chunk, embed, and batch-write with psycopg2
The model loads once, emits normalized 384-dimensional vectors, and writes them with `%s::vector`.

In [ ]:
if documents:
    rows = build_rows(documents, CHUNK_SIZE, CHUNK_OVERLAP)
    written = replace_embeddings(
        rows,
        [document["id"] for document in documents],
        BATCH_SIZE,
    )
    print(f"Wrote {written} chunk embeddings")
else:
    print("No new or changed documents need embeddings.")

## 8. Verify coverage, dimensions, and foreign-key integrity

In [ ]:
verification = lakebase.run_query(
    """
    SELECT
      (SELECT COUNT(*) FROM weather_documents) AS documents,
      (SELECT COUNT(*) FROM weather_embeddings) AS chunks,
      (SELECT COUNT(DISTINCT document_id) FROM weather_embeddings) AS embedded_documents,
      (SELECT COUNT(*) FROM weather_embeddings WHERE vector_dims(embedding) <> 384) AS wrong_dimensions,
      (SELECT COUNT(*)
         FROM weather_embeddings e
         LEFT JOIN weather_documents d ON d.id = e.document_id
        WHERE d.id IS NULL) AS orphan_embeddings
    """
)
display(verification)

if verification[0]["wrong_dimensions"] != 0 or verification[0]["orphan_embeddings"] != 0:
    raise RuntimeError("Vector-dimension or foreign-key verification failed")

## 9. Run a semantic-search smoke test
This uses the same model and pgvector `<=>` query as the Flask endpoint.

In [ ]:
if not SEARCH_QUERY:
    raise ValueError("search_query cannot be empty")
query_vector = vector_literal(embed_texts([SEARCH_QUERY])[0])
matches = lakebase.run_query(
    """
    SELECT d.location, d.source_type, d.headline, e.chunk_text,
           1 - (e.embedding <=> %s::vector) AS similarity
    FROM weather_embeddings e
    JOIN weather_documents d ON d.id = e.document_id
    ORDER BY e.embedding <=> %s::vector
    LIMIT 5
    """,
    (query_vector, query_vector),
)
print(f"Query: {SEARCH_QUERY}")
display(matches)

## 10. Ready to deploy
After the verification cells succeed, inspect both tables in Lakebase and capture the required evidence. Then create the Databricks App from this same Git repository. The app UI can sync additional documents and search the vectors; rerun this notebook after later syncs to embed newly added or changed documents.